### Libraries

In [1]:
import networkx as nx
import pandas as pd
import numpy as np

### Cargar grafo

In [2]:
def load_graph(edges_path):
    edges_df = pd.read_csv(edges_path, sep=" ", header=None, names=["from", "to"])

    # Eliminar duplicados (grafo no dirigido)
    edges_set = set(tuple(sorted(edge)) for edge in edges_df.to_numpy())
    unique_edges_df = pd.DataFrame(list(edges_set), columns=["source", "target"])

    G = nx.Graph()  # grafo no dirigido
    G.add_edges_from(unique_edges_df.to_numpy())

    return G

### Cargar círculos 

In [3]:
def load_circles(circles_path):
    circles = {}

    with open(circles_path, "r") as f:
        for i, line in enumerate(f):
            parts = line.strip().split()
            circles[f"circle{i}"] = list(map(int, parts[1:]))

    return circles

In [4]:
def summarize_circles(G, circles_dict):
    nodes = list(G.nodes())
    node_idx = {node: i for i, node in enumerate(nodes)}
    circle_names = list(circles_dict.keys())

    membership_matrix = np.zeros((len(nodes), len(circles_dict)), dtype=int)

    for j, c in enumerate(circle_names):
        for node in circles_dict[c]:
            if int(node) in node_idx:
                membership_matrix[node_idx[int(node)], j] = 1

    num_in_circle = membership_matrix.sum(axis=1)

    print("=== Circle Summary ===")
    print(f"Total circles: {len(circles_dict)}")
    print("Nodes in at least one circle:", (num_in_circle > 0).sum())
    print("Nodes in multiple circles:", (num_in_circle > 1).sum())
    print("Nodes not in any circle:", (num_in_circle == 0).sum())

    return membership_matrix


def filter_significant_circles(G, circles_dict, min_percent=5):
    total_nodes = len(G.nodes())
    all_nodes = set(map(int, G.nodes()))

    circle_sizes = {
        circle_name: len(members)
        for circle_name, members in circles_dict.items()
    }

    circle_summary_df = pd.DataFrame.from_dict(
        circle_sizes,
        orient="index",
        columns=["num_nodes"]
    )

    circle_summary_df["percent_nodes"] = (
        circle_summary_df["num_nodes"] / total_nodes * 100
    )

    circle_summary_df = circle_summary_df.sort_values(
        "num_nodes",
        ascending=False
    )

    circle_summary_df["percent_nodes"] = circle_summary_df["percent_nodes"].round(2)

    significant_circles = circle_summary_df[
        circle_summary_df["percent_nodes"] >= min_percent
    ].index.tolist()

    filtered_circles = {
        k: list(map(int, circles_dict[k]))
        for k in significant_circles
    }

    node_to_circles = {}

    for circle_name, members in filtered_circles.items():
        for node in members:
            node_to_circles.setdefault(node, []).append(circle_name)

    assigned_nodes = set(node_to_circles.keys())
    unassigned_nodes = all_nodes - assigned_nodes

    filtered_circles["circleNA"] = list(unassigned_nodes)

    filtered_summary_df = pd.DataFrame.from_dict(
        {
            k: len(v)
            for k, v in filtered_circles.items()
        },
        orient="index",
        columns=["num_nodes"]
    )

    filtered_summary_df["percent_nodes"] = (
        filtered_summary_df["num_nodes"] / total_nodes * 100
    ).round(2)

    print("\n=== Significant Circles ===")
    print(f"Minimum size: {min_percent}% of nodes")
    print(f"Remaining circles including circleNA: {len(filtered_circles)}")
    print(filtered_summary_df)

    return filtered_circles, circle_summary_df, filtered_summary_df

### Construir matriz ARD

In [5]:
def compute_ard_matrix(G, circles_dict):
    """
    ARD: filas = nodos, columnas = círculos.
    Valor = número de vecinos del nodo que están en ese círculo.
    """
    nodes = list(G.nodes())
    circle_names = list(circles_dict.keys())

    ard_matrix = pd.DataFrame(0, index=nodes, columns=circle_names)

    for node in nodes:
        neighbors = list(G.neighbors(node))

        for circle_name in circle_names:
            members = circles_dict[circle_name]
            count = sum(neigh in members for neigh in neighbors)
            ard_matrix.at[node, circle_name] = count

    return ard_matrix

In [6]:
# validación de ARD: la suma de cada fila debe ser igual al grado ajustado del nodo (teniendo en cuenta overlaps)
def validate_ard(G, ard_matrix, circles_dict):
    neighbors_dict = {node: list(G.neighbors(node)) for node in G.nodes()}

    # Cuántos círculos tiene cada nodo (permite overlap)
    circle_membership_counts = pd.Series(0, index=list(G.nodes()), dtype=int)

    circles_full_int = {
        k: [int(n) for n in v if int(n) in G.nodes()]
        for k, v in circles_dict.items()
    }

    for members in circles_full_int.values():
        for node in members:
            circle_membership_counts[node] += 1

    # Grado ajustado (teniendo en cuenta overlaps)
    adjusted_degree = pd.Series(0, index=list(G.nodes()), dtype=int)

    for node, neighbors in neighbors_dict.items():
        adjusted_degree[node] = sum(
            circle_membership_counts.get(neigh, 0)
            for neigh in neighbors
        )

    ard_sums = ard_matrix.sum(axis=1)
    matches = ard_sums == adjusted_degree

    print("¿ARD correcta (con overlap)?:", matches.all())

    return matches

### Membership matrix

In [7]:
def compute_membership_matrix(G, circles_dict):
    node_ids = list(map(str, G.nodes()))
    circle_names = list(circles_dict.keys())

    membership_matrix = pd.DataFrame(0, index=node_ids, columns=circle_names)

    for circle_name, members in circles_dict.items():
        str_members = list(map(str, members))
        valid_members = [m for m in str_members if m in membership_matrix.index]
        membership_matrix.loc[valid_members, circle_name] = 1

    return membership_matrix

In [ ]:
def compute_total_prop(G, circles_dict, membership_matrix):
    total_ties = len(G.edges())

    edges = [(str(u), str(v)) for u, v in G.edges()]

    feature_ties = np.zeros(len(circles_dict))

    for k, circle_name in enumerate(circles_dict.keys()):
        feature_ties[k] = sum(
            (membership_matrix.loc[u, circle_name] == 1) or
            (membership_matrix.loc[v, circle_name] == 1)
            for u, v in edges
        )

    total_prop = feature_ties.sum() / total_ties if total_ties != 0 else 0

    return total_prop

# total_prop =
# nº de veces que las aristas “tocan” algún círculo
# /
# nº total de aristas del grafo

### Uso

In [9]:
edges_path = "0.edges"
circles_path = "0.circles"

G = load_graph(edges_path)
circles = load_circles(circles_path)

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

print("=== Graph Info ===")
print(f"Nodos: {num_nodes}")
print(f"Aristas: {num_edges}")

# --- Circle summary original ---
membership_matrix_original = summarize_circles(G, circles)

# --- Filter significant circles and create circleNA ---
filtered_circles, circle_summary_df, filtered_summary_df = filter_significant_circles(
    G,
    circles,
    min_percent=5
)

# --- ARD using filtered circles ---
ard_matrix = compute_ard_matrix(G, filtered_circles)

print("\n=== ARD Matrix Info ===")
print(f"Dimensión ARD: {ard_matrix.shape}")
print(f"Nº círculos usados en ARD: {len(filtered_circles)}")

validate_ard(G, ard_matrix, filtered_circles)

ard_matrix.head()

=== Graph Info ===
Nodos: 333
Aristas: 2519
=== Circle Summary ===
Total circles: 24
Nodes in at least one circle: 277
Nodes in multiple circles: 39
Nodes not in any circle: 56

=== Significant Circles ===
Minimum size: 5% of nodes
Remaining circles including circleNA: 7
          num_nodes  percent_nodes
circle15        133          39.94
circle16         32           9.61
circle11         30           9.01
circle0          20           6.01
circle6          20           6.01
circle4          17           5.11
circleNA        107          32.13

=== ARD Matrix Info ===
Dimensión ARD: (333, 7)
Nº círculos usados en ARD: 7
¿ARD correcta (con overlap)?: True


,circle15,circle16,circle11,circle0,circle6,circle4,circleNA
14,0,0,1,0,8,0,5
312,0,0,1,0,9,0,15
67,53,1,16,1,0,9,2
324,14,0,7,0,0,0,4
113,28,1,8,2,0,2,0


In [10]:
# Crear membership matrix
membership_matrix = compute_membership_matrix(G, filtered_circles)

# Calcular total_prop
total_prop = compute_total_prop(G, filtered_circles, membership_matrix)

print("Total prop:", total_prop)

# Rutas locales de salida
ard_matrix_file_path = "ard_matrix_facebook0.csv"
membership_matrix_file_path = "membership_matrix_facebook0.csv"
total_prop_file_path = "total_prop_facebook0.txt"

# Guardar archivos
ard_matrix.to_csv(ard_matrix_file_path)
membership_matrix.to_csv(membership_matrix_file_path)
circle_summary_df.to_csv("circle_summary_facebook0.csv", index=False)
with open(total_prop_file_path, "w") as f:
    f.write(f"Total Proportion: {total_prop}\n")

print("Archivos guardados correctamente.")


Total prop: 1.5859468042874156
Archivos guardados correctamente.


### PASO 3 - FEATURES ALIGNED

In [11]:
edges_path = "0.edges"

G = load_graph(edges_path)

print("Graph loaded")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Graph loaded
Nodes: 333
Edges: 2519


In [12]:
features_path = "0.feat"
featnames_path = "0.featnames"

features_df = pd.read_csv(features_path, sep=" ", header=None)
featnames = pd.read_csv(featnames_path, sep=" ", header=None)

print("Features shape:", features_df.shape)
print("Featnames shape:", featnames.shape)

features_df.head()

Features shape: (347, 225)
Featnames shape: (224, 4)


,0,1,2,3,4,5,6,7,8,9,...,215,216,217,218,219,220,221,222,223,224
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
def get_features_for_graph(G, features_df):
    """
    Filter and reorder features_df to match exactly the nodes in G.

    Assumption:
    - First column of features_df is the node ID.
    - Remaining columns are binary/node features.
    """
    graph_nodes = list(G.nodes())

    features_df = features_df.copy()

    # First column = node ID
    features_df.iloc[:, 0] = features_df.iloc[:, 0].astype(int)
    features_df = features_df.set_index(features_df.columns[0])

    # Keep only nodes from G and preserve G node order
    features_aligned = features_df.loc[graph_nodes]
    features_aligned = features_aligned.reindex(graph_nodes)

    return features_aligned

In [14]:
features_aligned = get_features_for_graph(G, features_df)

print("Graph nodes:", G.number_of_nodes())
print("Features aligned shape:", features_aligned.shape)

assert features_aligned.shape[0] == G.number_of_nodes(), \
    "Mismatch: features rows must match graph nodes"

features_aligned.head()

Graph nodes: 333
Features aligned shape: (333, 224)


,1,2,3,4,5,6,7,8,9,10,...,215,216,217,218,219,220,221,222,223,224
0,,,,,,,,,,,,,,,,,,,,,
14,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
312,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
67,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,1,0,0,0,0
324,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
113,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [15]:
features_aligned.to_csv("features_aligned_facebook0.csv")

print("Saved: features_aligned_facebook0.csv")

Saved: features_aligned_facebook0.csv
